In [2]:
import sys
!{sys.executable} -m pip install streamlit


[notice] A new release of pip available: 22.2.2 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import requests
import pandas as pd
import streamlit as st
import matplotlib.pyplot as plt

#   Fetch Data from FPL API  
@st.cache_data
def load_fpl_data():
    url = "https://fantasy.premierleague.com/api/bootstrap-static/"
    response = requests.get(url)
    data = response.json()

    players = pd.DataFrame(data['elements'])
    teams = pd.DataFrame(data['teams'])
    positions = pd.DataFrame(data['element_types'])

    # Map team and position names
    team_map = dict(zip(teams['id'], teams['name']))
    position_map = dict(zip(positions['id'], positions['singular_name']))

    players['Team'] = players['team'].map(team_map)
    players['Position'] = players['element_type'].map(position_map)
    players['Name'] = players['first_name'] + ' ' + players['second_name']
    players['Cost (£M)'] = players['now_cost'] / 10
    players['ROI'] = players['total_points'] / players['Cost (£M)']

    return players[['Name', 'Team', 'Position', 'Cost (£M)', 'total_points', 'ROI']]

#   Load Data  
df = load_fpl_data()

#   Streamlit UI  
st.set_page_config(page_title="FPL ROI Calculator", layout="wide")
st.title("Fantasy Premier League ROI Calculator")
st.markdown("Compare player value based on fantasy points per £1M cost.")

#   Filters  
positions = ['All'] + sorted(df['Position'].unique())
selected_position = st.selectbox("Filter by Position", positions)

if selected_position != 'All':
    filtered_df = df[df['Position'] == selected_position]
else:
    filtered_df = df

#   Display Table  
st.subheader("Top ROI Players")
st.dataframe(filtered_df.sort_values(by='ROI', ascending=False).reset_index(drop=True))

#   Plot ROI  
top_roi = filtered_df.sort_values(by='ROI', ascending=False).head(10)

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(top_roi['Name'], top_roi['ROI'], color='mediumseagreen')
ax.set_xlabel('ROI (Points per £1M)')
ax.set_title('Top 10 ROI Players')
ax.invert_yaxis()
st.pyplot(fig)

2025-09-10 19:30:30.450 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2025-09-10 19:30:30.461 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2025-09-10 19:30:30.462 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-09-10 19:30:31.279 
  command:

    streamlit run C:\Users\jeeya\AppData\Roaming\Python\Python310\site-packages\ipykernel_launcher.py [ARGUMENTS]
2025-09-10 19:30:31.287 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-09-10 19:30:31.295 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-09-10 19:30:31.298 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-09-10 19:30:31.805 Thread '

DeltaGenerator()